In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.data_quality_passed.upi_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA QUALITY PASSED UPI DATA
# ============================================================

source_df = spark.read.table(
    "fintech360.data_quality_passed.upi_data"
)

# ============================================================
# 2. STANDARDIZE BUSINESS DIMENSIONS
# ============================================================

df = (source_df
    .withColumn("business_payer_vpa", F.lower(F.trim(F.col("payer_vpa"))))
    .withColumn("business_payee_vpa", F.lower(F.trim(F.col("payee_vpa"))))
    .withColumn("business_merchant_category", F.upper(F.trim(F.col("merchant_category"))))
    .withColumn("business_currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("business_upi_app", F.upper(F.trim(F.col("upi_app"))))
    .withColumn("business_device_os", F.upper(F.trim(F.col("device_os"))))
    .withColumn("business_account_type", F.upper(F.trim(F.col("payer_account_type"))))
    .withColumn("business_transaction_amount", F.col("transaction_amount").cast("double"))
    .withColumn("business_transaction_status", F.col("transaction_status").cast("int"))
    .withColumn("business_is_refund", F.col("is_refund").cast("int"))
)

# ============================================================
# 3. TRANSACTION DATE AND TIME FEATURES
# transaction_ts is Unix epoch seconds.
# ============================================================

df = (df
    .withColumn(
        "business_transaction_datetime",
        F.to_timestamp(
            F.from_unixtime(F.col("transaction_ts").cast("long"))
        )
    )
    .withColumn(
        "business_transaction_date",
        F.to_date("business_transaction_datetime")
    )
    .withColumn(
        "business_transaction_month",
        F.date_format("business_transaction_datetime", "yyyy-MM")
    )
    .withColumn(
        "business_transaction_year",
        F.year("business_transaction_datetime")
    )
    .withColumn(
        "business_transaction_hour",
        F.hour("business_transaction_datetime")
    )
    .withColumn(
        "business_day_of_week",
        F.date_format("business_transaction_datetime", "EEEE")
    )
    .withColumn(
        "business_is_weekend",
        F.when(
            F.dayofweek("business_transaction_datetime").isin([1, 7]),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 4. TRANSACTION STATUS
# Assumption: 1 = SUCCESS, 0 = FAILED.
# ============================================================

df = df.withColumn(
    "business_transaction_status",
    F.when(
        F.col("business_transaction_status") == 1, "SUCCESS"
    ).when(
        F.col("business_transaction_status") == 0, "FAILED"
    ).otherwise("UNKNOWN")
)

df = df.withColumn(
    "successful_transaction_flag",
    F.when(F.col("business_transaction_status") == "SUCCESS", 1)
     .otherwise(0)
)

df = df.withColumn(
    "failed_transaction_flag",
    F.when(F.col("business_transaction_status") == "FAILED", 1)
     .otherwise(0)
)

# ============================================================
# 5. SUCCESSFUL PAYMENT AMOUNT
# Failed transactions should not contribute to successful spend.
# ============================================================

df = (df
    .withColumn(
        "successful_transaction_amount",
        F.when(
            F.col("successful_transaction_flag") == 1,
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "failed_transaction_amount",
        F.when(
            F.col("failed_transaction_flag") == 1,
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
)

# ============================================================
# 6. REFUND FEATURES
# This identifies flagged refund transactions.
# It does not assume the original payment has been matched.
# ============================================================

df = (df
    .withColumn(
        "refund_flag",
        F.when(F.col("business_is_refund") == 1, 1).otherwise(0)
    )
    .withColumn(
        "successful_refund_flag",
        F.when(
            (F.col("business_is_refund") == 1) &
            (F.col("successful_transaction_flag") == 1),
            1
        ).otherwise(0)
    )
    .withColumn(
        "successful_refund_amount",
        F.when(
            F.col("successful_refund_flag") == 1,
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "successful_non_refund_amount",
        F.when(
            (F.col("successful_transaction_flag") == 1) &
            (F.col("business_is_refund") == 0),
            F.col("business_transaction_amount")
        ).otherwise(F.lit(0.0))
    )
)

# ============================================================
# 7. TRANSACTION VALUE SEGMENTATION
# ============================================================

df = df.withColumn(
    "transaction_amount_band",
    F.when(
        F.col("business_transaction_amount") < 100,
        "01_BELOW_100"
    ).when(
        F.col("business_transaction_amount") < 500,
        "02_100_TO_499"
    ).when(
        F.col("business_transaction_amount") < 1000,
        "03_500_TO_999"
    ).when(
        F.col("business_transaction_amount") < 5000,
        "04_1000_TO_4999"
    ).otherwise("05_5000_PLUS")
)

# ============================================================
# 8. MERCHANT CATEGORY GROUPING
# Preserve original category and derive a broader segment.
# ============================================================

df = df.withColumn(
    "business_spending_group",
    F.when(
        F.col("business_merchant_category").isin(
            "GROCERY", "FOOD", "ENTERTAINMENT", "HEALTHCARE"
        ),
        "CONSUMER_SPENDING"
    ).when(
        F.col("business_merchant_category") == "PERSONAL_TRANSFER",
        "PERSONAL_TRANSFER"
    ).when(
        F.col("business_merchant_category").isNull() |
        (F.col("business_merchant_category") == ""),
        "UNKNOWN"
    ).otherwise("OTHER_SPENDING")
)

# ============================================================
# 9. PAYER AND PAYEE FEATURES
# ============================================================

df = (df
    .withColumn(
        "payer_vpa_handle",
        F.element_at(F.split(F.col("business_payer_vpa"), "@"), -1)
    )
    .withColumn(
        "payee_vpa_handle",
        F.element_at(F.split(F.col("business_payee_vpa"), "@"), -1)
    )
    .withColumn(
        "same_payer_payee_vpa_flag",
        F.when(
            F.col("business_payer_vpa") == F.col("business_payee_vpa"),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 10. PAYMENT CHANNEL AND ACCOUNT SEGMENTS
# ============================================================

df = (df
    .withColumn(
        "upi_channel",
        F.when(
            F.col("business_upi_app").isNull() |
            (F.col("business_upi_app") == ""),
            "UNKNOWN"
        ).otherwise(F.col("business_upi_app"))
    )
    .withColumn(
        "device_segment",
        F.when(
            F.col("business_device_os") == "ANDROID", "ANDROID_DEVICE"
        ).when(
            F.col("business_device_os") == "IOS", "IOS_DEVICE"
        ).otherwise("OTHER_OR_UNKNOWN_DEVICE")
    )
    .withColumn(
        "payer_account_segment",
        F.when(
            F.col("business_account_type") == "SAVINGS", "SAVINGS_ACCOUNT"
        ).when(
            F.col("business_account_type") == "CURRENT", "CURRENT_ACCOUNT"
        ).when(
            F.col("business_account_type") == "WALLET", "WALLET_ACCOUNT"
        ).otherwise("OTHER_OR_UNKNOWN_ACCOUNT")
    )
)

# ============================================================
# 11. BUSINESS REVIEW FLAGS
# These are analytical indicators, not fraud conclusions.
# ============================================================

df = (df
    .withColumn(
        "high_value_transaction_flag",
        F.when(
            F.col("business_transaction_amount") >= 5000, 1
        ).otherwise(0)
    )
    .withColumn(
        "low_value_transaction_flag",
        F.when(
            F.col("business_transaction_amount") < 100, 1
        ).otherwise(0)
    )
    .withColumn(
        "transaction_review_signal",
        F.when(
            F.col("same_payer_payee_vpa_flag") == 1,
            "SAME_PAYER_PAYEE"
        ).when(
            F.col("failed_transaction_flag") == 1,
            "FAILED_TRANSACTION"
        ).when(
            F.col("successful_refund_flag") == 1,
            "SUCCESSFUL_REFUND_REQUIRES_RECONCILIATION"
        ).otherwise("NO_BASIC_REVIEW_SIGNAL")
    )
)

# ============================================================
# 12. TRANSFORMATION METADATA
# ============================================================

df = df.withColumn(
    "business_transformed_at",
    F.current_timestamp()
)

# ============================================================
# 13. FINAL BUSINESS DATAFRAME
# ============================================================

upi_data_business = df

display(upi_data_business.limit(10))


In [0]:

# Create destination schema if it does not exist
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.business_transformation
""")

# Append transformed UPI transactions
(upi_data_business.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.business_transformation.upi_data"))

print("UPI business transformation completed.")
print("Target: fintech360.business_transformation.upi_data")
